# 🚀 Apigee X Provisioning - Global Control Plane (PSC or VPC Peering)

<a href="https://colab.research.google.com/github/luiscuellarh/apigee-unified-ai-gateway/blob/main/01_apigee_provisioning_global.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

End-to-end provisioning of an **Apigee X** organization, runtime instance, and external HTTPS Load Balancer (`nip.io`) using either **Private Service Connect (`PSC`)** or **VPC Peering (`VPC_PEERING`)**.

> [!WARNING]
> **`BILLING_TYPE` (`EVALUATION`, `PAYG`, `SUBSCRIPTION`) and `NETWORKING_MODE` (`PSC`, `VPC_PEERING`) are permanent once the organization is created and cannot be changed later without deleting the organization.**

**Estimated total run time:** ~30–45 minutes (safe to re-run at any point).


## Step 0: Colab Authentication & Configuration

Authenticates your Google Cloud session, validates project access up front, and sets all deployment parameters (~10 seconds).


In [ ]:
#@title ⚙️ Configuration Form [Run this cell]
#@markdown Fill in the parameters below and choose **Runtime > Run all** (fields set to `"auto"` or left blank derive their value automatically):

import os
import sys
import json
import time
import subprocess
import requests

try:
    from google.colab import auth
    auth.authenticate_user()
    print("✅ Authenticated with Google Cloud in Colab.")
except ImportError:
    print("ℹ️ Using local Google Cloud credentials.")

#@markdown #### 1. Google Cloud Project & Region
#@markdown *Leave `ANALYTICS_REGION` as `"auto"` to use `REGION`.*
PROJECT_ID = "your-project-id" #@param {type:"string"}
REGION = "europe-west1" #@param ["europe-west1", "europe-west3", "europe-west4", "us-central1", "us-east1", "us-east4", "us-west1", "asia-east1", "asia-northeast1", "asia-southeast1", "australia-southeast1"]
ANALYTICS_REGION = "auto" #@param ["auto", "europe-west1", "europe-west3", "europe-west4", "us-central1", "us-east1", "us-east4", "us-west1", "asia-east1", "asia-northeast1", "asia-southeast1", "australia-southeast1"]

#@markdown #### 2. Billing Plan (Permanent at Org Creation)
BILLING_TYPE = "EVALUATION" #@param ["EVALUATION", "PAYG", "SUBSCRIPTION"]
RUNTIME_TYPE = "CLOUD" #@param ["CLOUD", "HYBRID"]

#@markdown #### 3. Networking Mode & VPC Configuration
#@markdown *`NETWORKING_MODE` is fixed at org creation. `NETWORK_NAME`, `SUBNET_NAME`, and `SUBNET_CIDR` apply to both modes.*
NETWORKING_MODE = "PSC" #@param ["PSC", "VPC_PEERING"]
NETWORK_NAME = "apigee-network" #@param {type:"string"}
SUBNET_NAME = "apigee-subnet" #@param {type:"string"}
SUBNET_CIDR = "10.10.0.0/24" #@param {type:"string"}
#@markdown ---
#@markdown **VPC Peering Mode Only** *(used only when `NETWORKING_MODE == "VPC_PEERING"`; ignored in `PSC` mode)*:
PEERING_CIDR_SIZE = "SLASH_22" #@param ["SLASH_22", "SLASH_28"]
PEERING_RANGE_NAME = "apigee-peering-range" #@param {type:"string"}
SUPPORT_RANGE_NAME = "apigee-support-range" #@param {type:"string"}

# Derive prefix lengths from PEERING_CIDR_SIZE
PEERING_PREFIX_LEN = 22 if PEERING_CIDR_SIZE == "SLASH_22" else 28
SUPPORT_PREFIX_LEN = 28

#@markdown #### 4. Apigee Environment
#@markdown *Leave `INSTANCE_NAME` (`"auto"` -> `apigee-<REGION>`) or `ENVGROUP_NAME` (`"auto"` -> `<ENV_NAME>-group`) as `"auto"` or blank to derive them automatically.*
INSTANCE_NAME = "auto" #@param {type:"string"}
ENV_NAME = "dev" #@param {type:"string"}
ENVGROUP_NAME = "auto" #@param {type:"string"}
ENABLE_MONETIZATION = True #@param {type:"boolean"}
ENABLE_API_SECURITY = True #@param {type:"boolean"}
ENABLE_ADVANCED_APIOPS = True #@param {type:"boolean"}

# Fallback derivation for fields left blank or set to "auto"
if not ANALYTICS_REGION.strip() or ANALYTICS_REGION.strip() == "auto":
    ANALYTICS_REGION = REGION
if not SUBNET_NAME.strip() or SUBNET_NAME.strip() == "auto":
    SUBNET_NAME = f"{NETWORK_NAME}-{REGION}"
if not INSTANCE_NAME.strip() or INSTANCE_NAME.strip() == "auto":
    INSTANCE_NAME = f"apigee-{REGION}"
if not ENVGROUP_NAME.strip() or ENVGROUP_NAME.strip() == "auto":
    ENVGROUP_NAME = f"{ENV_NAME}-group"

# Fail-early validation
if PROJECT_ID == "your-project-id" or not PROJECT_ID.strip():
    raise ValueError("❌ Set a valid PROJECT_ID in Step 0 before running.")
if NETWORKING_MODE not in ("PSC", "VPC_PEERING"):
    raise ValueError(f"❌ Invalid NETWORKING_MODE '{NETWORKING_MODE}'. Choose 'PSC' or 'VPC_PEERING'.")
if PEERING_CIDR_SIZE not in ("SLASH_22", "SLASH_28"):
    raise ValueError(f"❌ Invalid PEERING_CIDR_SIZE '{PEERING_CIDR_SIZE}'. Choose 'SLASH_22' or 'SLASH_28'.")

proj_check = subprocess.run(
    ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectId)"],
    capture_output=True, text=True
)
if proj_check.returncode != 0:
    raise RuntimeError(f"❌ Cannot access project '{PROJECT_ID}': {proj_check.stderr.strip()}")

subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID, "--quiet"], check=True)

for _k, _v in {
    "PROJECT_ID": PROJECT_ID,
    "REGION": REGION,
    "ANALYTICS_REGION": ANALYTICS_REGION,
    "BILLING_TYPE": BILLING_TYPE,
    "RUNTIME_TYPE": RUNTIME_TYPE,
    "NETWORKING_MODE": NETWORKING_MODE,
    "NETWORK_NAME": NETWORK_NAME,
    "SUBNET_NAME": SUBNET_NAME,
    "SUBNET_CIDR": SUBNET_CIDR,
    "PEERING_CIDR_SIZE": PEERING_CIDR_SIZE,
    "PEERING_RANGE_NAME": PEERING_RANGE_NAME,
    "SUPPORT_RANGE_NAME": SUPPORT_RANGE_NAME,
    "INSTANCE_NAME": INSTANCE_NAME,
    "ENV_NAME": ENV_NAME,
    "ENVGROUP_NAME": ENVGROUP_NAME,
    "ENABLE_MONETIZATION": str(ENABLE_MONETIZATION),
    "ENABLE_API_SECURITY": str(ENABLE_API_SECURITY),
    "ENABLE_ADVANCED_APIOPS": str(ENABLE_ADVANCED_APIOPS),
}.items():
    os.environ[_k] = _v

print(f"✅ Project verified: {PROJECT_ID} | Region: {REGION} (Analytics: {ANALYTICS_REGION}) | Billing: {BILLING_TYPE} ({RUNTIME_TYPE})")
print(f"✅ Mode: {NETWORKING_MODE} | Network: {NETWORK_NAME} | Subnet: {SUBNET_NAME} ({SUBNET_CIDR})")
print(f"✅ Instance: {INSTANCE_NAME} | Env: {ENV_NAME} ({ENVGROUP_NAME}) | Add-ons: Monetization={ENABLE_MONETIZATION}, Security={ENABLE_API_SECURITY}, AdvancedApiOps={ENABLE_ADVANCED_APIOPS}")
if NETWORKING_MODE == "VPC_PEERING":
    print(f"ℹ️ VPC Peering config: {PEERING_CIDR_SIZE} (/{PEERING_PREFIX_LEN}), ranges={PEERING_RANGE_NAME}, {SUPPORT_RANGE_NAME} (/{SUPPORT_PREFIX_LEN})")


In [ ]:
# Helper function for Google API requests and long-running operation polling
def get_auth_token():
    token = subprocess.check_output(
        ["gcloud", "auth", "print-access-token"], text=True
    ).strip()
    return token

def apigee_headers():
    return {
        "Authorization": f"Bearer {get_auth_token()}",
        "Content-Type": "application/json; charset=utf-8"
    }

def poll_apigee_operation(op_url, description="Operation", poll_interval=15, max_timeout=3600):
    start_time = time.time()
    print(f"⏳ Waiting for {description} to complete...")
    while time.time() - start_time < max_timeout:
        resp = requests.get(op_url, headers=apigee_headers())
        if resp.status_code == 200:
            data = resp.json()
            if data.get("done", False):
                if "error" in data:
                    raise RuntimeError(f"❌ {description} failed: {json.dumps(data['error'], indent=2)}")
                elapsed = int(time.time() - start_time)
                print(f"✅ {description} completed successfully in {elapsed // 60}m {elapsed % 60}s!")
                return data.get("response", {})
        elapsed = int(time.time() - start_time)
        mins, secs = divmod(elapsed, 60)
        print(f"   ⏱️ Still in progress... ({mins}m {secs:02d}s elapsed)", end="\r")
        time.sleep(poll_interval)
    raise TimeoutError(f"❌ Timed out waiting for {description} after {max_timeout // 60} minutes.")

print("🛠️ Apigee helper utilities initialized.")


## Step 1: Enable Google Cloud APIs

Enables the required Google Cloud APIs (`apigee`, `compute`, `apihub`, `cloudkms`, plus `servicenetworking` when in `VPC_PEERING` mode). Takes ~30–90 seconds.


In [ ]:
#@title 🔌 Enable APIs
apis_to_enable = [
    "apigee.googleapis.com",
    "compute.googleapis.com",
    "apihub.googleapis.com",
    "cloudkms.googleapis.com",
]
if NETWORKING_MODE == "VPC_PEERING":
    apis_to_enable.append("servicenetworking.googleapis.com")

print(f"⏳ Enabling APIs ({', '.join(apis_to_enable)}) — takes ~30–90s...")
subprocess.run(
    ["gcloud", "services", "enable", *apis_to_enable, f"--project={PROJECT_ID}", "--quiet"],
    check=True
)
print(f"✅ Enabled APIs: {', '.join(apis_to_enable)}")


## Step 2: Networking Mode — PSC or VPC Peering

Ensures the VPC network and regional subnet exist, and in `VPC_PEERING` mode verifies or reserves the global peering ranges and connects Service Networking. Takes ~15 seconds in `PSC` mode or ~1–3 minutes in `VPC_PEERING` mode.


In [ ]:
#@title 🌐 Networking Mode & VPC Configuration [Run this cell]
#@markdown Uses the configuration from **Step 0**.

_required = ["PROJECT_ID", "REGION", "NETWORKING_MODE", "NETWORK_NAME", "SUBNET_NAME", "SUBNET_CIDR", "PEERING_CIDR_SIZE", "PEERING_RANGE_NAME", "SUPPORT_RANGE_NAME"]
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise RuntimeError(f"❌ Missing configuration variable(s): {', '.join(_missing)}. Run Step 0 first.")

# 1. Verify or create VPC network
net_desc = subprocess.run(
    ["gcloud", "compute", "networks", "describe", NETWORK_NAME, f"--project={PROJECT_ID}", "--format=json"],
    capture_output=True, text=True
)
if net_desc.returncode == 0:
    print(f"ℹ️ VPC network '{NETWORK_NAME}' already exists (skipped).")
else:
    print(f"⏳ Creating custom-mode VPC network '{NETWORK_NAME}' (~15s)...")
    subprocess.run(
        ["gcloud", "compute", "networks", "create", NETWORK_NAME,
         f"--project={PROJECT_ID}", "--subnet-mode=custom", "--quiet"],
        check=True
    )
    print(f"✅ Created VPC network '{NETWORK_NAME}'.")

# 2. Verify existing regional subnet SUBNET_NAME in REGION or create it
subnets_proc = subprocess.run(
    ["gcloud", "compute", "networks", "subnets", "list",
     f"--network={NETWORK_NAME}", f"--regions={REGION}", f"--project={PROJECT_ID}", "--format=json"],
    capture_output=True, text=True, check=True
)
region_subnets = json.loads(subnets_proc.stdout or "[]")
matching_subnets = [s for s in region_subnets if s.get("name") == SUBNET_NAME]
if matching_subnets:
    DEPLOYED_SUBNET_CIDR = matching_subnets[0]["ipCidrRange"]
    print(f"ℹ️ Regional subnet '{SUBNET_NAME}' ({DEPLOYED_SUBNET_CIDR}) already exists in {REGION} (skipped).")
else:
    print(f"⏳ Creating regional subnet '{SUBNET_NAME}' ({SUBNET_CIDR}) in {REGION} (~15s)...")
    subprocess.run(
        ["gcloud", "compute", "networks", "subnets", "create", SUBNET_NAME,
         f"--network={NETWORK_NAME}", f"--region={REGION}", f"--range={SUBNET_CIDR}",
         f"--project={PROJECT_ID}", "--quiet"],
        check=True
    )
    DEPLOYED_SUBNET_CIDR = SUBNET_CIDR
    print(f"✅ Created regional subnet '{SUBNET_NAME}' ({DEPLOYED_SUBNET_CIDR}).")

os.environ["DEPLOYED_SUBNET_CIDR"] = DEPLOYED_SUBNET_CIDR

ALLOCATED_PEERING_CIDR = "N/A (PSC)"
ALLOCATED_SUPPORT_CIDR = "N/A (PSC)"
PEERING_STATE = "N/A (PSC)"

# 3. Mode-specific networking
if NETWORKING_MODE == "PSC":
    print("ℹ️ PSC mode — no peering ranges reserved, no Service Networking connection required.")

elif NETWORKING_MODE == "VPC_PEERING":
    for r_name, r_plen in [(PEERING_RANGE_NAME, PEERING_PREFIX_LEN), (SUPPORT_RANGE_NAME, SUPPORT_PREFIX_LEN)]:
        r_desc = subprocess.run(
            ["gcloud", "compute", "addresses", "describe", r_name, "--global", f"--project={PROJECT_ID}", "--format=json"],
            capture_output=True, text=True
        )
        if r_desc.returncode == 0:
            info = json.loads(r_desc.stdout)
            cidr = f"{info['address']}/{info['prefixLength']}"
            print(f"ℹ️ Global peering range '{r_name}' ({cidr}) already exists (skipped).")
        else:
            print(f"⏳ Reserving global peering range '{r_name}' (/{r_plen}) (~15s)...")
            subprocess.run(
                ["gcloud", "compute", "addresses", "create", r_name, "--global",
                 "--purpose=VPC_PEERING", f"--network={NETWORK_NAME}",
                 f"--prefix-length={r_plen}", f"--project={PROJECT_ID}", "--quiet"],
                check=True
            )
            r_after = subprocess.run(
                ["gcloud", "compute", "addresses", "describe", r_name, "--global", f"--project={PROJECT_ID}", "--format=json"],
                capture_output=True, text=True, check=True
            )
            info = json.loads(r_after.stdout)
            cidr = f"{info['address']}/{info['prefixLength']}"
            print(f"✅ Reserved global peering range '{r_name}': {cidr}")

        if r_name == PEERING_RANGE_NAME:
            ALLOCATED_PEERING_CIDR = cidr
        else:
            ALLOCATED_SUPPORT_CIDR = cidr

    ranges_arg = f"{PEERING_RANGE_NAME},{SUPPORT_RANGE_NAME}"
    print(f"⏳ Connecting Service Networking on '{NETWORK_NAME}' with [{ranges_arg}] (~1–3 min)...")
    conn_res = subprocess.run(
        ["gcloud", "services", "vpc-peerings", "connect",
         "--service=servicenetworking.googleapis.com", f"--network={NETWORK_NAME}",
         f"--ranges={ranges_arg}", f"--project={PROJECT_ID}", "--async", "--format=json"],
        capture_output=True, text=True
    )
    if conn_res.returncode != 0:
        print("ℹ️ Service Networking connection already exists — updating ranges with --force...")
        conn_res = subprocess.run(
            ["gcloud", "services", "vpc-peerings", "update",
             "--service=servicenetworking.googleapis.com", f"--network={NETWORK_NAME}",
             f"--ranges={ranges_arg}", "--force", f"--project={PROJECT_ID}", "--async", "--format=json"],
            capture_output=True, text=True, check=True
        )

    op_name = json.loads(conn_res.stdout or "{}").get("name")
    if op_name:
        op_id = op_name if op_name.startswith("operations/") else f"operations/{op_name}"
        op_url = f"https://servicenetworking.googleapis.com/v1/{op_id}"
        start_time = time.time()
        while time.time() - start_time < 1800:
            r = requests.get(op_url, headers=apigee_headers())
            if r.status_code == 200 and r.json().get("done", False):
                if "error" in r.json():
                    raise RuntimeError(f"❌ Service Networking peering failed: {json.dumps(r.json()['error'])}")
                break
            time.sleep(5)

    PEERING_STATE = "ACTIVE"
    print(f"✅ Service Networking peering is {PEERING_STATE} ({ALLOCATED_PEERING_CIDR}, {ALLOCATED_SUPPORT_CIDR}).")
else:
    raise ValueError(f"❌ Unrecognised NETWORKING_MODE: '{NETWORKING_MODE}'.")

os.environ["ALLOCATED_PEERING_CIDR"] = ALLOCATED_PEERING_CIDR
os.environ["ALLOCATED_SUPPORT_CIDR"] = ALLOCATED_SUPPORT_CIDR
os.environ["PEERING_STATE"] = PEERING_STATE


## Step 3: Create Apigee Organization (Global Control Plane)

Creates or verifies the Apigee Organization (`disableVpcPeering=true` for `PSC`, `disableVpcPeering=false` + `authorizedNetwork` for `VPC_PEERING`). Takes ~5–10 minutes when creating a new organization.


In [ ]:
#@title 🏛️ Create Apigee Organization
org_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}"
org_check = requests.get(org_url, headers=apigee_headers())

if org_check.status_code == 200 and org_check.json().get("state") == "ACTIVE":
    existing_org = org_check.json()
    raw_disabled = existing_org.get("disableVpcPeering", False)
    peering_disabled = raw_disabled is True or str(raw_disabled).lower() == "true"
    if NETWORKING_MODE == "PSC" and not peering_disabled:
        raise RuntimeError(
            f"❌ Existing Apigee Org '{PROJECT_ID}' has VPC Peering ENABLED (disableVpcPeering=False), "
            "which conflicts with NETWORKING_MODE='PSC'. Switch NETWORKING_MODE to 'VPC_PEERING' in Step 0 or recreate the org."
        )
    if NETWORKING_MODE == "VPC_PEERING" and peering_disabled:
        raise RuntimeError(
            f"❌ Existing Apigee Org '{PROJECT_ID}' was created in PSC-only mode (disableVpcPeering=True), "
            "which conflicts with NETWORKING_MODE='VPC_PEERING'. Switch NETWORKING_MODE to 'PSC' in Step 0 or recreate the org."
        )
    print(f"ℹ️ Apigee Organization '{PROJECT_ID}' already exists and matches {NETWORKING_MODE} mode (skipped).")
else:
    org_payload = {
        "displayName": PROJECT_ID,
        "description": f"Apigee Org for {PROJECT_ID}",
        "analyticsRegion": ANALYTICS_REGION,
        "runtimeType": RUNTIME_TYPE,
        "billingType": BILLING_TYPE,
        "addonsConfig": {
            "monetizationConfig": {"enabled": ENABLE_MONETIZATION},
            "advancedApiOpsConfig": {"enabled": ENABLE_ADVANCED_APIOPS},
            "apiSecurityConfig": {"enabled": ENABLE_API_SECURITY}
        },
        "portalDisabled": True
    }
    if NETWORKING_MODE == "PSC":
        org_payload["disableVpcPeering"] = "true"
    elif NETWORKING_MODE == "VPC_PEERING":
        org_payload["disableVpcPeering"] = "false"
        org_payload["authorizedNetwork"] = NETWORK_NAME
    else:
        raise ValueError(f"❌ Unrecognised NETWORKING_MODE: '{NETWORKING_MODE}'.")

    print(f"⏳ Creating Apigee Organization '{PROJECT_ID}' in {NETWORKING_MODE} mode (~5–10 min)...")
    create_resp = requests.post(
        f"https://apigee.googleapis.com/v1/organizations?parent=projects/{PROJECT_ID}",
        headers=apigee_headers(),
        json=org_payload
    )
    if create_resp.status_code in (200, 201):
        op_name = create_resp.json().get("name")
        if op_name:
            poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op_name}", "Organization Creation")
    elif "already exists" in create_resp.text:
        print("ℹ️ Organization creation already in progress — waiting for ACTIVE state...")
    else:
        raise RuntimeError(f"❌ Organization creation failed ({create_resp.status_code}): {create_resp.text}")

    start_time = time.time()
    while time.time() - start_time < 900:
        resp = requests.get(org_url, headers=apigee_headers())
        if resp.status_code == 200 and resp.json().get("state") == "ACTIVE":
            print(f"✅ Apigee Organization '{PROJECT_ID}' is ACTIVE.")
            break
        time.sleep(15)


## Step 4: Create Apigee Runtime Instance

Provisions the regional Apigee runtime instance and retrieves its `serviceAttachment` (`PSC`) or internal `host` IP (`VPC_PEERING`). Takes ~20–30 minutes when creating a new instance.


In [ ]:
#@title 💻 Provision Apigee Runtime Instance
instance_url = f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/instances/{INSTANCE_NAME}"
inst_resp = requests.get(instance_url, headers=apigee_headers())

TARGET_SERVICE = ""
INSTANCE_HOST = ""

if inst_resp.status_code == 200 and inst_resp.json().get("state") == "ACTIVE":
    inst_data = inst_resp.json()
    TARGET_SERVICE = inst_data.get("serviceAttachment", "")
    INSTANCE_HOST = inst_data.get("host", "")
    print(f"ℹ️ Instance '{INSTANCE_NAME}' already exists and is ACTIVE (skipped).")
else:
    inst_payload = {
        "name": INSTANCE_NAME,
        "location": REGION,
        "description": f"Apigee instance in {REGION}",
        "displayName": f"Instance {REGION}"
    }
    if NETWORKING_MODE == "VPC_PEERING":
        inst_payload["peeringCidrRange"] = PEERING_CIDR_SIZE

    print(f"⏳ Creating Apigee Instance '{INSTANCE_NAME}' in {REGION} (~20–30 min)...")
    create_inst = requests.post(
        f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/instances",
        headers=apigee_headers(),
        json=inst_payload
    )
    if create_inst.status_code in (200, 201):
        op_name = create_inst.json().get("name")
        if op_name:
            poll_apigee_operation(
                f"https://apigee.googleapis.com/v1/{op_name}",
                "Instance Creation",
                poll_interval=20,
                max_timeout=3600
            )
    elif "already exists" in create_inst.text:
        print("ℹ️ Instance creation already in progress — waiting for ACTIVE state...")
    else:
        raise RuntimeError(f"❌ Instance creation failed ({create_inst.status_code}): {create_inst.text}")

    for _ in range(60):
        inst_resp = requests.get(instance_url, headers=apigee_headers())
        if inst_resp.status_code == 200 and inst_resp.json().get("state") == "ACTIVE":
            inst_data = inst_resp.json()
            TARGET_SERVICE = inst_data.get("serviceAttachment", "")
            INSTANCE_HOST = inst_data.get("host", "")
            break
        time.sleep(15)

if NETWORKING_MODE == "PSC":
    if not TARGET_SERVICE:
        raise RuntimeError("❌ Failed to retrieve serviceAttachment from Apigee Instance in PSC mode.")
    print(f"✅ Service Attachment: {TARGET_SERVICE}")
elif NETWORKING_MODE == "VPC_PEERING":
    print(f"✅ Runtime Host IP: {INSTANCE_HOST}")
    if TARGET_SERVICE:
        print(f"ℹ️ Service Attachment: {TARGET_SERVICE}")

os.environ["TARGET_SERVICE"] = TARGET_SERVICE
os.environ["INSTANCE_HOST"] = INSTANCE_HOST


## Step 5: Provision External HTTPS Load Balancer

Reserves the global external IPv4 address, configures the `<ip>.nip.io` Google-managed SSL certificate, and provisions the PSC NEG and Global External HTTPS Load Balancer. Takes ~1–2 minutes.


In [ ]:
#@title 🛡️ Create Ingress & External HTTPS LB
def run_or_skip(desc_cmd, create_cmd, label):
    if subprocess.run(desc_cmd, capture_output=True, text=True).returncode == 0:
        print(f"ℹ️ {label} already exists (skipped).")
    else:
        subprocess.run(create_cmd, check=True)
        print(f"✅ Created {label}.")

print("⏳ Provisioning External HTTPS Load Balancer (~1–2 min)...")

# 1. Global IPv4 address
run_or_skip(
    ["gcloud", "compute", "addresses", "describe", "apigee-ipaddress", "--global", f"--project={PROJECT_ID}"],
    ["gcloud", "compute", "addresses", "create", "apigee-ipaddress", "--ip-version=IPV4", "--global", f"--project={PROJECT_ID}", "--quiet"],
    "Global IP 'apigee-ipaddress'"
)
IP_ADDRESS = subprocess.check_output(
    ["gcloud", "compute", "addresses", "describe", "apigee-ipaddress", "--format=value(address)", "--global", f"--project={PROJECT_ID}"],
    text=True
).strip()
RUNTIME_HOST_ALIAS = f"{IP_ADDRESS.replace('.', '-')}.nip.io"
os.environ["IP_ADDRESS"] = IP_ADDRESS
os.environ["RUNTIME_HOST_ALIAS"] = RUNTIME_HOST_ALIAS
print(f"✅ External IP: {IP_ADDRESS} | Hostname: {RUNTIME_HOST_ALIAS}")

# 2. Google-managed SSL Certificate
run_or_skip(
    ["gcloud", "compute", "ssl-certificates", "describe", "apigee-ssl-cert", f"--project={PROJECT_ID}"],
    ["gcloud", "compute", "ssl-certificates", "create", "apigee-ssl-cert", f"--domains={RUNTIME_HOST_ALIAS}", f"--project={PROJECT_ID}", "--quiet"],
    f"SSL certificate 'apigee-ssl-cert' ({RUNTIME_HOST_ALIAS})"
)

if TARGET_SERVICE:
    # 3. PSC NEG
    run_or_skip(
        ["gcloud", "compute", "network-endpoint-groups", "describe", "apigee-neg", f"--region={REGION}", f"--project={PROJECT_ID}"],
        ["gcloud", "compute", "network-endpoint-groups", "create", "apigee-neg",
         "--network-endpoint-type=private-service-connect", f"--psc-target-service={TARGET_SERVICE}",
         f"--region={REGION}", f"--network={NETWORK_NAME}", f"--subnet={SUBNET_NAME}", f"--project={PROJECT_ID}", "--quiet"],
        f"PSC NEG 'apigee-neg' ({REGION})"
    )

    # 4. Backend Service + attach NEG
    run_or_skip(
        ["gcloud", "compute", "backend-services", "describe", "apigee-backend", "--global", f"--project={PROJECT_ID}"],
        ["gcloud", "compute", "backend-services", "create", "apigee-backend",
         "--load-balancing-scheme=EXTERNAL_MANAGED", "--protocol=HTTPS", "--global", f"--project={PROJECT_ID}", "--quiet"],
        "Backend Service 'apigee-backend'"
    )
    bs_info = json.loads(subprocess.check_output(
        ["gcloud", "compute", "backend-services", "describe", "apigee-backend", "--global", f"--project={PROJECT_ID}", "--format=json"],
        text=True
    ))
    if any("apigee-neg" in b.get("group", "") for b in bs_info.get("backends", [])):
        print("ℹ️ PSC NEG 'apigee-neg' already attached to 'apigee-backend' (skipped).")
    else:
        subprocess.run(
            ["gcloud", "compute", "backend-services", "add-backend", "apigee-backend",
             "--network-endpoint-group=apigee-neg", f"--network-endpoint-group-region={REGION}",
             "--global", f"--project={PROJECT_ID}", "--quiet"],
            check=True
        )
        print("✅ Attached PSC NEG 'apigee-neg' to 'apigee-backend'.")

    # 5. URL Map, HTTPS Proxy, Global Forwarding Rule
    run_or_skip(
        ["gcloud", "compute", "url-maps", "describe", "apigee-lb", "--global", f"--project={PROJECT_ID}"],
        ["gcloud", "compute", "url-maps", "create", "apigee-lb", "--default-service=apigee-backend", "--global", f"--project={PROJECT_ID}", "--quiet"],
        "URL Map 'apigee-lb'"
    )
    run_or_skip(
        ["gcloud", "compute", "target-https-proxies", "describe", "apigee-proxy", f"--project={PROJECT_ID}"],
        ["gcloud", "compute", "target-https-proxies", "create", "apigee-proxy", "--url-map=apigee-lb", "--ssl-certificates=apigee-ssl-cert", f"--project={PROJECT_ID}", "--quiet"],
        "Target HTTPS Proxy 'apigee-proxy'"
    )
    run_or_skip(
        ["gcloud", "compute", "forwarding-rules", "describe", "apigee-fw-rule", "--global", f"--project={PROJECT_ID}"],
        ["gcloud", "compute", "forwarding-rules", "create", "apigee-fw-rule",
         "--load-balancing-scheme=EXTERNAL_MANAGED", "--network-tier=PREMIUM", f"--address={IP_ADDRESS}",
         "--target-https-proxy=apigee-proxy", "--ports=443", "--global", f"--project={PROJECT_ID}", "--quiet"],
        "Global Forwarding Rule 'apigee-fw-rule' (Port 443)"
    )
else:
    print(f"ℹ️ VPC_PEERING mode without serviceAttachment — runtime is reachable on VPC '{NETWORK_NAME}' at {INSTANCE_HOST}.")


## Step 6: Create Environment, Environment Group & Attachments

Creates the Environment Group, Environment, attachments to the runtime instance, and Cloud Trace configuration. Takes ~2–5 minutes.


In [ ]:
#@title 🔗 Configure Environment, Environment Group & Attachments
print(f"⏳ Configuring Environment '{ENV_NAME}' and Envgroup '{ENVGROUP_NAME}' (~2–5 min)...")

# 1. Environment Group
eg_resp = requests.post(
    f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/envgroups",
    headers=apigee_headers(),
    json={"name": ENVGROUP_NAME, "hostnames": [RUNTIME_HOST_ALIAS]}
)
if eg_resp.status_code in (200, 201):
    op = eg_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", f"Envgroup '{ENVGROUP_NAME}' Creation")
elif "already exists" in eg_resp.text:
    print(f"ℹ️ Envgroup '{ENVGROUP_NAME}' already exists (skipped).")
else:
    raise RuntimeError(f"❌ Envgroup creation failed ({eg_resp.status_code}): {eg_resp.text}")

# 2. Environment
env_resp = requests.post(
    f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/environments",
    headers=apigee_headers(),
    json={"name": ENV_NAME}
)
if env_resp.status_code in (200, 201):
    op = env_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", f"Environment '{ENV_NAME}' Creation")
elif "already exists" in env_resp.text:
    print(f"ℹ️ Environment '{ENV_NAME}' already exists (skipped).")
else:
    raise RuntimeError(f"❌ Environment creation failed ({env_resp.status_code}): {env_resp.text}")

# 3. Envgroup Attachment
eg_att_resp = requests.post(
    f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/envgroups/{ENVGROUP_NAME}/attachments",
    headers=apigee_headers(),
    json={"name": f"{ENVGROUP_NAME}-{ENV_NAME}", "environment": ENV_NAME}
)
if eg_att_resp.status_code in (200, 201):
    op = eg_att_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", "Envgroup Attachment")
elif "already exists" in eg_att_resp.text:
    print(f"ℹ️ Environment '{ENV_NAME}' already attached to '{ENVGROUP_NAME}' (skipped).")
else:
    raise RuntimeError(f"❌ Envgroup attachment failed ({eg_att_resp.status_code}): {eg_att_resp.text}")

# 4. Instance Attachment
inst_att_resp = requests.post(
    f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/instances/{INSTANCE_NAME}/attachments",
    headers=apigee_headers(),
    json={"environment": ENV_NAME}
)
if inst_att_resp.status_code in (200, 201):
    op = inst_att_resp.json().get("name")
    if op:
        poll_apigee_operation(f"https://apigee.googleapis.com/v1/{op}", "Instance Attachment")
elif "already exists" in inst_att_resp.text:
    print(f"ℹ️ Environment '{ENV_NAME}' already attached to '{INSTANCE_NAME}' (skipped).")
else:
    raise RuntimeError(f"❌ Instance attachment failed ({inst_att_resp.status_code}): {inst_att_resp.text}")

# 5. Cloud Trace
trace_resp = requests.patch(
    f"https://apigee.googleapis.com/v1/organizations/{PROJECT_ID}/environments/{ENV_NAME}/traceConfig",
    headers=apigee_headers(),
    json={
        "exporter": "CLOUD_TRACE",
        "endpoint": PROJECT_ID,
        "samplingConfig": {"sampler": "PROBABILITY", "samplingRate": 0.5}
    }
)
if trace_resp.status_code not in (200, 201):
    raise RuntimeError(f"❌ Cloud Trace config failed ({trace_resp.status_code}): {trace_resp.text}")
print("✅ Cloud Trace exporter configured.")


## Step 7: Summary & Verification

Prints the provisioned endpoints, regional resources, global reservations, and live SSL certificate status (~5 seconds; SSL certificate reaches `ACTIVE` 10–20 minutes after creation).


In [ ]:
#@title 🎉 Deployment Summary & Test
ssl_status_proc = subprocess.run(
    ["gcloud", "compute", "ssl-certificates", "describe", "apigee-ssl-cert",
     "--format=value(managed.status)", f"--project={PROJECT_ID}"],
    capture_output=True, text=True
)
ssl_status = ssl_status_proc.stdout.strip() if ssl_status_proc.returncode == 0 else "PROVISIONING"

print("=" * 68)
print("🎉 APIGEE PROVISIONING COMPLETE!")
print("=" * 68)
print(f"📌 Project / Org:      {PROJECT_ID}")
print(f"💳 Billing Type:       {BILLING_TYPE}")
print(f"🔌 Networking Mode:    {NETWORKING_MODE}")
print(f"🌿 Environment:        {ENV_NAME} ({ENVGROUP_NAME})")
print(f"🔗 Public IP:          {IP_ADDRESS}")
print(f"🌍 Hostname (nip.io):  https://{RUNTIME_HOST_ALIAS}")
print(f"🔒 SSL Cert Status:    {ssl_status} (transitions to ACTIVE in ~10–20 min)")
print("-" * 68)
print("🏗️ DEPLOYED (regional)")
print("-" * 68)
print(f"🌐 VPC Network:        {NETWORK_NAME}")
print(f"📍 Regional Subnet:    {SUBNET_NAME} ({DEPLOYED_SUBNET_CIDR}) in {REGION}")
print(f"📍 Apigee Instance:    {INSTANCE_NAME} ({REGION})")
if NETWORKING_MODE == "PSC":
    print(f"🔗 Service Attachment: {TARGET_SERVICE}")
else:
    print(f"🖥️ Runtime Host IP:    {INSTANCE_HOST or 'N/A'}")
print("-" * 68)
print("📦 RESERVED (global)")
print("-" * 68)
if NETWORKING_MODE == "VPC_PEERING":
    print(f"📐 Runtime CIDR Range: {PEERING_RANGE_NAME} -> {ALLOCATED_PEERING_CIDR} ({PEERING_CIDR_SIZE})")
    print(f"📐 Support CIDR Range: {SUPPORT_RANGE_NAME} -> {ALLOCATED_SUPPORT_CIDR}")
    print(f"🤝 VPC Peering State:  {PEERING_STATE}")
else:
    print("ℹ️ None (PSC mode — no global VPC peering ranges reserved)")
print("=" * 68)
print(f"👉 Apigee Console: https://console.cloud.google.com/apigee/proxies?project={PROJECT_ID}")
